---
title: "Analisis de calificaciones"
author: "Irvin"
format:
  html:
    theme: cosmo
    css: styles.css
    toc: true
    toc-depth: 2
    number-sections: true
jupyter: python3
execute:
  warning: false
  error: false
  cache: false
---


In [9]:
%%html

<script>
  window.__define_backup = window.define;
  window.define = undefined;
</script>

<script src="https://cdn.jsdelivr.net/npm/papaparse@5.4.1/papaparse.min.js"></script>
<script src="https://cdn.plot.ly/plotly-2.32.0.min.js"></script>

<script>
  window.define = window.__define_backup;
</script>

<div style="font-family: Arial, sans-serif; margin-bottom:12px;">
  <b>Sube aquí tu archivo CSV:</b><br>
  <input id="csvFileInput" type="file" accept=".csv" />
  <span id="status" style="margin-left:12px;color:#555;"></span>
</div>
<div id="plotContainer" style="margin-top:20px;"></div>

<style>
#plotContainer { width: 100%; max-width: 1100px; min-height: 300px; }
.progress { font-size: 0.9em; color:#666; margin-top:8px; }
</style>

<script>
/* Conversión segura a número */
function toNumber(val) {
  if (val === null || val === undefined) return NaN;
  const s = String(val).trim().replace(/\s+/g,'').replace(',', '.');
  return parseFloat(s);
}

/* Encuentra índices de columnas esperadas */
function findColumns(headers) {
  const lower = headers.map(h => h.trim().toLowerCase());
  const idxNombre = lower.indexOf('nombre');
  const candidates = [
    'calificación/10.00','calificacion/10.00',
    'calificación','calificacion',
    'calificación/10','calificacion/10'
  ];
  let idxCalif = -1;
  for (const c of candidates) {
    const i = lower.indexOf(c);
    if (i !== -1) { idxCalif = i; break; }
  }
  return { idxNombre, idxCalif };
}

/* Dibuja gráfica con Plotly */
function drawPlot(names, values, title) {
  const plotDiv = document.getElementById('plotContainer');
  plotDiv.innerHTML = '';
  const data = [{ x: names, y: values, type: 'bar', marker: { color: '#4CAF50' } }];
  const layout = {
    title: title || 'Nombre vs Calificación/10.00',
    xaxis: { tickangle: -45, automargin: true },
    yaxis: { title: 'Calificación / 10.00' },
    margin: { t: 60, b: 160 },
    height: 520
  };
  Plotly.newPlot('plotContainer', data, layout, {responsive: true});
}

/* Manejo del archivo */
document.getElementById('csvFileInput').addEventListener('change', function(e) {
  const file = e.target.files[0];
  const status = document.getElementById('status');
  const plotDiv = document.getElementById('plotContainer');
  plotDiv.innerHTML = '';
  if (!file) { status.textContent = ''; return; }

  // Verificamos que las librerías existan en el entorno global
  if (typeof Plotly === 'undefined' || typeof Papa === 'undefined') {
      plotDiv.innerHTML = '<p style="color:darkred;">Error: Las librerías Plotly o PapaParse no se cargaron correctamente.</p>';
      return;
  }

  status.textContent = 'Parseando con PapaParse...';
  const TOP_N = 100;
  const AGGREGATE_DUP = true;
  const agg = new Map();
  let totalRows = 0;
  let headers = null;

  Papa.parse(file, {
    header: true,
    skipEmptyLines: true,
    worker: true,
    chunkSize: 1024 * 1024,
    chunk: function(results, parser) {
      if (!headers && results.meta && results.meta.fields) {
        headers = results.meta.fields;
        const cols = findColumns(headers);
        if (cols.idxNombre === -1 || cols.idxCalif === -1) {
          parser.abort();
          plotDiv.innerHTML = '<p style="color:darkred;">El CSV debe contener las columnas <b>Nombre</b> y <b>Calificación/10.00</b>.</p>';
          status.textContent = '';
          return;
        }
      }
      for (const row of results.data) {
        totalRows++;
        const keys = Object.keys(row);
        const nombreKey = keys[findColumns(keys).idxNombre];
        const califKey = keys[findColumns(keys).idxCalif];
        
        const nombre = row[nombreKey] ?? row[keys[0]];
        const califRaw = row[califKey] ?? row[keys[findColumns(keys).idxCalif] ?? 0];
        const califNum = toNumber(califRaw);
        
        if (nombre === undefined || nombre === null) continue;
        if (isNaN(califNum)) continue;
        
        const key = String(nombre).trim();
        if (AGGREGATE_DUP) {
          const cur = agg.get(key);
          if (!cur) agg.set(key, { sum: califNum, count: 1 });
          else { cur.sum += califNum; cur.count += 1; }
        } else {
          agg.set(key + '___' + totalRows, { sum: califNum, count: 1, label: key });
        }
      }
      if (totalRows % 5000 === 0) status.textContent = `Procesadas ~${totalRows} filas...`;
    },
    complete: function() {
      status.textContent = 'Finalizado. Preparando gráfica...';
      const items = [];
      for (const [k, v] of agg.entries()) {
        const label = v.label || k;
        const value = (v.sum / v.count);
        items.push({ name: label, val: value });
      }
      if (items.length === 0) {
        plotDiv.innerHTML = '<p style="color:darkred;">No hay datos válidos para graficar.</p>';
        status.textContent = '';
        return;
      }
      items.sort((a,b) => b.val - a.val);
      const top = items.slice(0, TOP_N);
      drawPlot(top.map(x=>x.name), top.map(x=>x.val), file.name + ' (top ' + top.length + ')');
      status.textContent = '';
    },
    error: function(err) {
      status.textContent = '';
      plotDiv.innerHTML = '<p style="color:darkred;">Error al leer el archivo con PapaParse: ' + String(err) + '</p>';
    }
  });
});
</script>